## Chat Templates
Chat templates are the foundation of instruction tuning - they provide a consistent format for structuring interactions between language models, users, and external tools. Think of them as the “grammar” that teaches models how to understand conversations, distinguish between different speakers, and respond appropriately.

### Base Models vs Instruct Models
First, we need to understand the difference between base and instruct models. This is crucial for effective fine-tuning.

- Base Model (SmolLM3-3B-Base): Trained on raw text to predict the next token. If you give it “The weather today is”, it might continue with “sunny and warm” or any plausible continuation.

- Instruct Model (SmolLM3-3B): Fine-tuned to follow instructions and engage in conversations. If you ask “What’s the weather like?”, it understands this as a question requiring a response as a new message.

### The Transformation Process
The journey from base to instruct model involves:

- Chat template: A structured format for interactions between language models, users, and external tools.
- Supervised fine-tuning: The technique used to train the model to generate appropriate responses.
SmolLM3 uses the ChatML (Chat Markup Language) format, which has become a standard in the industry due to its clarity and flexibility.

In the next chapter, we will go in to preference alignment. This is a technique that allows you to fine-tune a model to generate responses that are preferred by a human.

### Pipeline Usage: Automated Chat Processing
The easiest way to use an open source large language model is to use the pipeline abstraction in 🤗 Transformers. It handles chat templates seamlessly, making it easy to use chat models without manual template management. So much so, you won’t even need to know the chat template format.

In [1]:
  import torch
  import psutil
  import os

  def get_memory_usage():
      """Get current memory usage in GB"""
      process = psutil.Process(os.getpid())
      mem_info = process.memory_info()

      stats = {
          'ram_gb': mem_info.rss / (1024**3),
      }

      if torch.cuda.is_available():
          stats['gpu_allocated_gb'] = torch.cuda.memory_allocated() / (1024**3)
          stats['gpu_reserved_gb'] = torch.cuda.memory_reserved() / (1024**3)
      elif torch.backends.mps.is_available():
          stats['mps_allocated_gb'] = torch.mps.current_allocated_memory() / (1024**3)

      return stats

  # Before generation
  print("Memory before loading model:")
  print(get_memory_usage())

Memory before loading model:
{'ram_gb': 0.18804931640625, 'mps_allocated_gb': 0.0}


In [2]:
import os

# Disable progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from transformers import pipeline

pipe = pipeline("text-generation", "HuggingFaceTB/SmolLM3-3B")

print("\nMemory after loading model:")
print(get_memory_usage())

# define the conversation

messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting"},
]

response = pipe(messages, max_new_tokens=128, temperature=0.7)
print("response: ", response)
print(response[0]['generated_text'][-1]) # print the assistant's response

Device set to use mps:0



Memory after loading model:
{'ram_gb': 0.026153564453125, 'mps_allocated_gb': 5.72781777381897}


/Users/abhisheksingh/Desktop/ML_problems/.venv/lib/python3.10/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


response:  [{'generated_text': [{'role': 'system', 'content': 'You are a friendly chatbot who always responds in the style of a pirate'}, {'role': 'user', 'content': 'How many helicopters can a human eat in one sitting'}, {'role': 'assistant', 'content': "<think>\nOkay, the user is asking how many helicopters a human can eat in one sitting. That's a pretty strange question. Let me break it down.\n\nFirst, helicopters are heavy machinery. They're not edible. They're made of metal, plastic, and other materials that the human body can't digest. So, physically, a human can't eat a helicopter. But maybe the user is being playful or is looking for a creative answer.\n\nIf we take it literally, the answer is zero. A human can't eat a helicopter because it's not food. But if we're considering it in a hypothetical or absurd sense, maybe"}]}]
{'role': 'assistant', 'content': "<think>\nOkay, the user is asking how many helicopters a human can eat in one sitting. That's a pretty strange question. 

In [4]:
import gc

# Multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a helpful math tutor."},
    {"role": "user", "content": "Can you help me with calculus?"},
]

def profile_generation(max_tokens):
    gc.collect()
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()
    elif torch.backends.mps.is_available():
      torch.mps.empty_cache()
    
    # Get baseline
    baseline = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    
    # Generate
    response = pipe(conversation, max_new_tokens=max_tokens, temperature=0.8, do_sample=True)
    
    # Get peak
    peak = psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    
    # Count tokens generated
    generated_text = response[0]['generated_text'][-1]['content']
    
    stats = {
      'max_tokens': max_tokens,
      'baseline_gb': baseline,
      'peak_gb': peak,
      'increase_gb': peak - baseline,
    }
    
    if torch.cuda.is_available():
      stats['gpu_peak_gb'] = torch.cuda.max_memory_allocated() / (1024**3)
    elif torch.backends.mps.is_available():
      stats['mps_peak_gb'] = torch.mps.current_allocated_memory() / (1024**3)
    
    return stats

print("Testing with max_new_tokens=200")
stats_200 = profile_generation(200)
for k, v in stats_200.items():
  print(f"  {k}: {v}")

print("\nTesting with max_new_tokens=2000")
stats_2000 = profile_generation(2000)
for k, v in stats_2000.items():
  print(f"  {k}: {v}")

print(f"\nMemory difference: {stats_2000['increase_gb'] - stats_200['increase_gb']:.3f} GB")


Testing with max_new_tokens=200
  max_tokens: 200
  baseline_gb: 0.20709228515625
  peak_gb: 0.5428619384765625
  increase_gb: 0.3357696533203125
  mps_peak_gb: 5.72781777381897

Testing with max_new_tokens=2000
  max_tokens: 2000
  baseline_gb: 0.57818603515625
  peak_gb: 0.292510986328125
  increase_gb: -0.285675048828125
  mps_peak_gb: 5.72781777381897

Memory difference: -0.621 GB


In [6]:
# configure generation parameters

generation_config = {
    "max_new_tokens": 200,
    "temperature": 0.8,
    "do_sample": True,
    "top_p": 0.9,
    "repetition_penalty": 1.1
}

# Multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a helpful math tutor."},
    {"role": "user", "content": "Can you help me with calculus?"},
]

response = pipe(conversation, **generation_config)
conversation = response[0]['generated_text']

# Continue the conversation
conversation.append({"role": "user", "content": "What is a derivative?"})
response = pipe(conversation, **generation_config)

print("Final conversation:")
for message in response[0]['generated_text']:
    print(f"{message['role']}: {message['content']}")

python(17362,0x20aab4800) malloc: Failed to allocate segment from range group - out of space
python(17362,0x20aab4800) malloc: Failed to allocate segment from range group - out of space


Final conversation:
system: You are a helpful math tutor.
user: Can you help me with calculus?
assistant: <think>
Okay, the user asked for help with calculus. Let me start by understanding what they need. Calculus is a broad subject that includes topics like differentiation and integration, limits, sequences, series, multivariable calculus, differential equations, and more. I should ask them to specify which area of calculus they're struggling with so I can provide targeted assistance.

First, I'll acknowledge their request and express willingness to help. Then, prompt them to mention the specific topic or problem they have in mind. It's important to keep my response friendly and encouraging. Maybe they're stuck on derivatives, integrals, or something else. By knowing exactly where they need help, I can give precise explanations and examples tailored to their needs.

I should also consider possible follow-up questions. For instance, if they mention differentials or optimization problem

### Working with SmolLM3 Chat Templates in Code
The transformers library automatically handles chat template formatting through the tokenizer. This means you only need to structure your messages correctly, and the library takes care of the special token formatting. Here’s how to work with SmolLM3’s chat template:

In [7]:
from transformers import AutoTokenizer

# Load SmolLM3's tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

# Structure your conversation as a list of message dictionaries
messages = [
    {"role": "system", "content": "You are a helpful assistant focused on technical topics."},
    {"role": "user", "content": "Can you explain what a chat template is?"},
    {"role": "assistant", "content": "A chat template structures conversations between users and AI models by providing a consistent format that helps the model understand different roles and maintain context."}
]

# Apply the chat template
formatted_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False, # return string instead of tokens
    add_generation_prompt=True # add prompt for next assistant response
)

print(formatted_chat)

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 06 November 2025
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant focused on technical topics.

<|im_start|>user
Can you explain what a chat template is?<|im_end|>
<|im_start|>assistant
A chat template structures conversations between users and AI models by providing a consistent format that helps the model understand different roles and maintain context.<|im_end|>
<|im_start|>assistant



### Understanding the Message Structure
Each message in the conversation follows a simple dictionary format:

- role: Identifies who is speaking (system, user, assistant, or tool).
- content: The actual message content.

### Message Types:

- System Messages: Set behavior and context for the entire conversation
- User Messages: Questions, requests, or statements from the human user
- Assistant Messages: Responses from the AI model
- Tool Messages: Results from function calls (for advanced use cases)
- System Messages: Setting the Context
- System messages are crucial for controlling SmolLM3’s behavior. They act as persistent instructions that influence all subsequent interactions. To create a system message, you can use the system role and the content key:



In [8]:
# Professional assistant
system_message = {
    "role": "system",
    "content": "You are a professional customer service agent. Always be polite, clear, and helpful."
}

# Technical expert
system_message = {
    "role": "system",
    "content": "You are a senior software engineer with excellent understanding of network engineering. Provide detailed technical explanations with code examples when appropriate."
}


### Generation Prompts: Controlling Model Behavior
One of the most important concepts in chat templates is the generation prompt. This tells the model when it should start generating a response versus continuing existing text.



In [9]:
# The add_generation_prompt parameter controls whether the template adds tokens that indicate the start of a bot response:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

messages = [
    {"role": "user", "content": "Hi there!"},
    {"role": "assistant", "content": "Nice to meet you!"},
    {"role": "user", "content": "Can I ask a question?"}
]

# without generation prompt - for completed conversations

formatted_without = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("Without generation prompt:")
print(formatted_without)
print("\n" + "="*50 + "\n")

# With generation prompt - for inference
formatted_with = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

print("With generation prompt:")
print(formatted_with)


Without generation prompt:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 06 November 2025
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any e

The generation prompt ensures that when the model generates text, it will write a bot response instead of doing something unexpected like continuing the user’s message.

### When to Use Generation Prompts
- For inference: Use add_generation_prompt=True when you want the model to generate a response.
- For training: Use add_generation_prompt=False when preparing training data with complete conversations.
- For evaluation: Use add_generation_prompt=True to test model responses.

### Continuing Final Messages: Advanced Response Control
The continue_final_message parameter allows you to make the model continue the last message in a conversation instead of starting a new one. This is particularly useful for “prefilling” responses or ensuring specific output formats.

In [10]:
# Prefill a JSON response
chat = [
    {"role": "user", "content": "Can you format the answer in JSON?"},
    {"role": "assistant", "content": '{"name": "'},
]

# Continue the final message
formatted_chat = tokenizer.apply_chat_template(
    chat, 
    tokenize=False, 
    continue_final_message=True
)

print("Continuing final message:")
print(formatted_chat)
print("\n" + "="*50 + "\n")

# Compare with starting a new message
formatted_new = tokenizer.apply_chat_template(
    chat, 
    tokenize=False,
    add_generation_prompt=True
)

print("Starting new message:")
print(formatted_new)

Continuing final message:
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 06 November 2025
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any er

## Chat Template Parameters: `add_generation_prompt` vs `continue_final_message`

#### add_generation_prompt=True


This is the standard parameter for typical chat or instruction-following tasks.


* What it does: It appends the special tokens that signal the start of the assistant's turn at the end of the formatted conversation. For example, if a model's format
 is [INST] user_message [/INST] assistant_message, this flag adds the [/INST] part, prompting the model to generate the assistant's message.
* Goal: To get a reply from the model in response to the last message in the conversation history.
* Use Case: You have a complete conversation history (e.g., system prompt, user message) and you want the model to generate the next turn in the conversation.


Example:
If your input is [{"role": "user", "content": "Hello"}], using add_generation_prompt=True would format it into something like:
"<start_token>user\nHello<end_token>assistant\n"
The model sees the assistant prompt and knows it's time to start generating its own message.

---

#### continue_final_message=True


This is a more specialized parameter used for completion tasks.


* What it does: It formats the conversation but intentionally omits the end-of-message token for the very last message in the list.
* Goal: To have the model continue or complete the text of the final message, rather than replying to it.
* Use Case: You provide an incomplete message and want the model to finish it. This is useful for tasks like text completion, story writing, or having the model expand
 on a thought you started.


Example:
If your input is [{"role": "assistant", "content": "The first rule of thermodynamics is"}], using continue_final_message=True would format it into something like:
"<start_token>assistant\nThe first rule of thermodynamics is"
The model sees the incomplete sentence and its task is to continue writing from that exact point, without adding a conversational reply.



In [11]:
# View the actual template
print("SmolLM3 Chat Template:")
print(tokenizer.chat_template)

# See what special tokens are used
print("\nSpecial tokens:")
print(f"BOS: {tokenizer.bos_token}")
print(f"EOS: {tokenizer.eos_token}")
print(f"UNK: {tokenizer.unk_token}")
print(f"PAD: {tokenizer.pad_token}")

# Check for custom tokens
special_tokens = tokenizer.special_tokens_map
for name, token in special_tokens.items():
    print(f"{name}: {token}")

SmolLM3 Chat Template:
{# ───── defaults ───── #}
{%- if enable_thinking is not defined -%}
{%- set enable_thinking = true -%}
{%- endif -%}

{# ───── reasoning mode ───── #}
{%- if enable_thinking -%}
  {%- set reasoning_mode = "/think" -%}
{%- else -%}
  {%- set reasoning_mode = "/no_think" -%}
{%- endif -%}

{# ───── header (system message) ───── #}
{{- "<|im_start|>system\n" -}}

{%- if messages[0].role == "system" -%}
  {%- set system_message = messages[0].content -%}
  {%- if "/no_think" in system_message -%}
    {%- set reasoning_mode = "/no_think" -%}
  {%- elif "/think" in system_message -%}
    {%- set reasoning_mode = "/think" -%}
  {%- endif -%}
  {%- set custom_instructions = system_message.replace("/no_think", "").replace("/think", "").rstrip() -%}
{%- endif -%}

{%- if "/system_override" in system_message -%}
  {{- custom_instructions.replace("/system_override", "").rstrip() -}}
  {{- "<|im_end|>\n" -}}
{%- else -%}
  {{- "## Metadata\n\n" -}}
  {{- "Knowledge Cutoff Dat

In [12]:
def debug_chat_template(messages, tokenizer):
    """Debug chat template application"""
    
    # Apply template
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and decode to see actual tokens
    tokens = tokenizer(formatted, return_tensors="pt")
    
    print("=== TEMPLATE DEBUG ===")
    print(f"Input messages: {len(messages)}")
    print(f"Formatted length: {len(formatted)} chars")
    print(f"Token count: {tokens['input_ids'].shape[1]}")
    print("\nFormatted text:")
    print(repr(formatted))  # Shows escape characters
    print("\nTokens:")
    print(tokens['input_ids'][0].tolist()[:20], "...")  # First 20 tokens
    print("\nDecoded tokens:")
    for i, token_id in enumerate(tokens['input_ids'][0][:20]):
        token = tokenizer.decode([token_id])
        print(f"{i:2d}: {token_id:5d} -> {repr(token)}")

# Example usage
debug_messages = [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi there!"}
]

debug_chat_template(debug_messages, tokenizer)

=== TEMPLATE DEBUG ===
Input messages: 2
Formatted length: 1404 chars
Token count: 258

Formatted text:
'<|im_start|>system\n## Metadata\n\nKnowledge Cutoff Date: June 2025\nToday Date: 06 November 2025\nReasoning Mode: /think\n\n## Custom Instructions\n\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant finding

# Supervised Fine Tuning

## Supervised Fine-Tuning with SmolLM3

Supervised Fine-Tuning (SFT) is the cornerstone of instruction tuning - it's how we transform a base language model into an instruction-following assistant. In this section, you'll learn to fine-tune **SmolLM3** using real-world datasets and production-ready tools.

### What is Supervised Fine-Tuning?

SFT is the process of continuing to train a pre-trained model on task-specific datasets with labeled examples. Think of it as specialized education:

- Pre-training teaches the model general language understanding (like learning to read).
- Supervised fine-tuning teaches specific skills and behaviors (like learning to do a specific task).

The key insight behind SFT is that we're not teaching the model new knowledge from scratch. Instead, we're **reshaping how existing knowledge is applied**. The pre-trained model already understands language, grammar, and has absorbed vast amounts of factual information. SFT focuses this general capability toward specific application patterns, response styles, and task-specific requirements.

This approach is effective because it leverages the rich representations learned during pre-training while requiring significantly less computational resources than training from scratch. The model learns to recognize instruction patterns, maintain conversation context, follow safety guidelines, and generate responses in desired formats.

> [!TIP]
> Before starting SFT, consider whether using an existing instruction-tuned model with well-crafted prompts would suffice for your use case. SFT involves significant computational resources and engineering effort, so it should only be pursued when prompting existing models proves insufficient. Learn more about this decision process in the [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter11/3).

#### The SmolLM3 SFT Journey

SmolLM3's instruction-following capabilities come from a sophisticated SFT process:

1. **Base Model** (`SmolLM3-3B-Base`): Trained on 11T tokens of general text
2. **SFT Training**: Fine-tuned on curated instruction datasets including SmolTalk2
3. **Preference Alignment**: Further refined using techniques like APO (Anchored Preference Optimization)

This multi-stage approach creates a model that's both knowledgeable and helpful.

### Why SFT Works: The Science Behind It

SFT is effective because it leverages the rich representations learned during pre-training while adapting the model's behavior patterns. During SFT, the model's parameters are fine-tuned through gradient descent on task-specific examples, causing subtle but important changes in how the model processes and generates text.

Specifically, the process works through several key mechanisms:

**Behavioral Adaptation**: The model learns to recognize instruction patterns and respond appropriately. This involves updating the attention mechanisms to focus on instruction cues in language and adjusting the output distribution to favor the desired responses. Research has shown that instruction tuning primarily affects the model's surface-level behavior rather than its underlying knowledge [(Wei et al., 2021)](https://huggingface.co/papers/2109.01652).

**Task Specialization**: Rather than learning entirely new concepts, the model learns to apply its existing knowledge in specific contexts. This is why SFT is much more efficient than pre-training - we're refining existing capabilities rather than building them from scratch. Studies indicate that most of the factual knowledge comes from pre-training, while SFT teaches the model how to format and present this knowledge appropriately [(Ouyang et al., 2022)](https://huggingface.co/papers/2203.02155).

**Safety Alignment**: Through exposure to carefully curated examples, the model learns to be more helpful, harmless, and honest. This involves both learning what to say and what not to say in various situations. The effectiveness of this approach has been demonstrated in works like InstructGPT [(Ouyang et al., 2022)](https://huggingface.co/papers/2203.02155) and Constitutional AI [(Bai et al., 2022)](https://huggingface.co/papers/2204.05862).

> [!TIP]
> SFT doesn't teach new facts - it teaches new behaviors. The model already knows about the world from pre-training; SFT teaches it how to be a helpful assistant using that knowledge.

The mathematical foundation involves minimizing the cross-entropy loss between the model's predictions and the target responses in your training dataset. This process gradually shifts the model's probability distributions to favor the types of responses demonstrated in your training examples.

### When to Use Supervised Fine-Tuning

The key question is: "Does my use case require behavior that differs significantly from general-purpose conversation?" If yes, SFT is likely beneficial.

Decision framework: Use this checklist to determine if SFT is appropriate for your project:
- Have you tried prompt engineering with existing instruction-tuned models?
- Do you need consistent output formats that prompting cannot achieve?
- Is your domain specialized enough that general models struggle?
- Do you have high-quality training data (at least 1,000 examples)?
- Do you have the computational resources for training and evaluation?

If you answered "yes" to most of these, SFT is likely worth pursuing.

### The SFT Process

Now let's move on to the process of SFT itself. The SFT process follows a systematic approach that ensures high-quality results:

#### 1. Dataset Preparation and Selection

The quality of your training data is the most critical factor for successful SFT. Unlike pre-training where quantity often matters most, SFT prioritizes quality and relevance. Your dataset should contain input-output pairs that demonstrate exactly the behavior you want your model to learn.

**Choose the Right Dataset**:
- SmolTalk2: The dataset used to train SmolLM3, containing high-quality instruction-response pairs.
- Domain-specific datasets: For specialized applications (medical, legal, technical).
- Custom datasets: Your own curated examples for specific use cases.

Each training example should consist of:
1. **Input prompt**: The user's instruction or question
2. **Expected response**: The ideal assistant response
3. **Context** (optional): Any additional information needed


> [!TIP]
> Dataset size guidelines:
> - Minimum: 1,000 high-quality examples for basic fine-tuning.
> - Recommended: 10,000+ examples for robust performance.
> - Quality over quantity: 1,000 well-curated examples often outperform 10,000 mediocre ones.
>
> Remember: Your model will learn to mimic the patterns in your training data, so invest time in data curation.

#### 2. Environment Setup and Configuration

To set up an environment for SFT, we will need advance compute resources. We have three main options:

1. **Local GPU**: If you are lucky enough to have a access to a GPU with (at least 16GB of VRAM), you can train your model locally!
2. **Hugging Face Jobs**: If you don't have a GPU and don't want to use a cloud provider, you can use Hugging Face Jobs! We'll go into more detail about this in the [next section](./5).
3. **Notebook GPUs**: If you like to use a notebook provider like Google Colab, you can use their GPUs!
4. **Cloud GPU**: If you want to take control of your compute resources, you can use a cloud provider like AWS, GCP, or Azure.

In terms of hardware requirements, you will need a GPU with at least 16GB of VRAM, for example an Nvidia RTX 4080 or A10G.

#### 3. Training Configuration

Choosing the right hyperparameters is crucial for successful SFT. The goal is to find the sweet spot where the model learns effectively without overfitting or becoming unstable. Here's a detailed breakdown of each parameter and how to choose them:

**Key Hyperparameters**:

**Learning Rate** (5e-5 to 1e-4): Controls how much the model weights change with each update
- Start with 5e-5 for SmolLM3; this is conservative and stable.
- Too high: The model becomes unstable; loss oscillates or explodes.
- Too low: The model learns very slowly and may not converge in reasonable time.

**Batch Size** (4-16): Number of examples processed simultaneously
- Larger batches: More stable gradients, but require more GPU memory.
- Smaller batches: Less memory usage, but noisier gradients.
- Use gradient accumulation to achieve larger effective batch sizes.

**Max Sequence Length** (2048-4096): Maximum tokens per training example
- Longer sequences: Can handle more complex conversations.
- Shorter sequences: Faster training, less memory usage.
- Match your use case: Use the typical length of your target conversations.

**Training Steps** (1000-5000): Total number of parameter updates
- Depends on dataset size: More data usually requires more steps.
- Monitor validation loss: Stop when it stops improving.
- Rule of thumb: Three to five epochs through your dataset.

**Warmup Steps** (10% of total): Gradual learning rate increase at start
- Prevents early instability: Helps the model adapt gradually.
- Typical range: 100-500 steps for most SFT tasks.

> [!TIP]
> Hyperparameter starting points for SmolLM3:
>
> To bootstrap your training, you can use the following hyperparameters:
>
> **Learning Rate**:
>
> ```python
> # Conservative (stable, slower)
> learning_rate = 5e-5
>
> # Balanced (recommended)
> learning_rate = 1e-4
>
> # Aggressive (faster, less stable)
> learning_rate = 2e-4
> ```
>
> **Batch Size**:
>
> We can reduce GPU device batch size by using gradient accumulation.
>
> ```python
> # Limited GPU Memory
> per_device_train_batch_size = 2
> gradient_accumulation_steps = 8
>
> # Balanced GPU Memory
> per_device_train_batch_size = 4
> gradient_accumulation_steps = 4
>
> # More GPU Memory
> per_device_train_batch_size = 8
> gradient_accumulation_steps = 2
> ```
>
> **Max Sequence Length**:
>
> ```python
> # Very short sequences
> max_length = 512
>
> # Short sequences
> max_length = 1024
>
> # Long sequences 
> max_length = 2048
>
> # Very long sequences
> max_length = 4096
> ```

#### 4. Monitoring and Evaluation

Effective monitoring is crucial for successful SFT. Unlike pre-training where you primarily watch loss decrease, SFT requires careful attention to both quantitative metrics and qualitative outputs. The goal is to ensure your model is learning the desired behaviors without overfitting or developing unwanted patterns.

**Key Metrics to Monitor**:

**Training Loss**: Should decrease steadily but not too rapidly
- Healthy pattern: Smooth, gradual decrease.
- Warning signs: Sudden spikes, oscillations, or plateaus.
- Typical range: Starts around 2-4, should decrease to 0.5-1.5.

**Validation Loss**: Most important metric for preventing overfitting
- Should track training loss: A small gap indicates good generalization.
- Growing gap: Sign of overfitting; the model may be memorizing training data.
- Use for early stopping: Stop training when validation loss stops improving.

**Sample Outputs**: Regular qualitative checks are essential
- Generate responses: Test the model on held-out prompts during training.
- Check format consistency: Ensure the model follows desired response patterns.
- Monitor for degradation: Watch for repetitive or nonsensical outputs.

**Resource Usage**: Track GPU memory and training speed
- Memory spikes: May indicate batch size is too large.
- Slow training: Could suggest inefficient data loading or processing.

## Understanding Loss Patterns in SFT

Training loss typically follows three distinct phases, as illustrated in this example from the [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter11/3):

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/nlp_course_sft_loss_graphic.png" alt="SFT Training Progress" width="80%"/>
</div>

1. **Initial Sharp Drop**: Rapid adaptation to new data distribution
2. **Gradual Stabilization**: Learning rate slows as model fine-tunes  
3. **Convergence**: Loss values stabilize, indicating training completion

**Healthy Training Pattern**: The key indicator of successful training is a small gap between training and validation loss, suggesting the model is learning generalizable patterns rather than memorizing specific examples.

### Warning Signs to Watch For

Several patterns in the loss curves can indicate potential issues:

#### Overfitting Pattern

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/smol-course/images/resolve/main/sft_loss_1.png" alt="SFT Overfitting Pattern" width="80%"/>
</div>

If validation loss increases while training loss continues to decrease, your model is overfitting. Consider:
- Reducing training steps or epochs
- Increasing dataset size or diversity
- Adding regularization techniques
- Using early stopping based on validation loss

#### Underfitting Pattern  

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/sft_loss_2.png" alt="SFT Underfitting Pattern" width="80%"/>
</div>

If loss doesn't show significant improvement, the model might be:
- Learning too slowly (try increasing learning rate)
- Struggling with task complexity (check data quality)
- Hitting architectural limitations (consider different model size)

#### Potential Memorization

<div class="flex justify-center">
<img src="https://huggingface.co/datasets/smol-course/images/resolve/main/sft_loss_3.png" alt="SFT Memorization Pattern" width="80%"/>
</div>

Extremely low loss values could suggest memorization rather than learning. This is concerning if:
- Model performs poorly on new, similar examples
- Outputs lack diversity or creativity
- Responses are too similar to training examples

> [!TIP]
> Learn more about loss interpretation in the [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter11/3).

**Experiment Tracking with Trackio**:
For comprehensive experiment tracking, we recommend **[Trackio](https://huggingface.co/docs/trackio)** - a lightweight, free experiment tracking library built on Hugging Face infrastructure. Trackio provides:

- Drop-in replacement: API compatible with `wandb.init`, `wandb.log`, and `wandb.finish`.
- Local-first design: Dashboard runs locally by default, with optional Hugging Face Spaces hosting.
- Free hosting: Everything, including hosting on Hugging Face Spaces, is free.
- Lightweight: Fewer than 3,000 lines of Python code, easily extensible.

We can track any metrics during training, for example:

```python
# Simple Trackio integration
import trackio

# Initialize tracking
trackio.init(project="smollm3-sft")

# Log metrics during training
trackio.log({"train_in_loss": 0.5, "learning_rate": 5e-5})

# Finish tracking
trackio.finish()
```

The most convenient way to track your training is to use trackio's `transformers` integration. You can specify your Trackio project name and space ID using environment variables:

```bash
export TRACKIO_PROJECT_NAME="my-project"
export TRACKIO_SPACE_ID="username/space_id"
```

Or you can set them in your code:

```python
import os

os.environ["TRACKIO_PROJECT_NAME"] = "my-project"
os.environ["TRACKIO_SPACE_ID"] = "username/space_id"
```

Then you can use the `SFTTrainer` class from TRL to track your training and let it handle the tracking for you.

```python
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=config,
)
```

Trackio will serve an application with the metrics from training that looks like this:

<iframe src="https://trl-lib-trackio.hf.space/?project=trl-documentation&metrics=train/loss,train/mean_token_accuracy,train/num_tokens&sidebar=hidden" style="width: 100%; min-width: 300px; max-width: 800px;" height="830" frameBorder="0"></iframe>

### Logged metrics

While training and evaluating we record the following reward metrics:

* `global_step`: The total number of optimizer steps taken so far.
* `epoch`: The current epoch number, based on dataset iteration.
* `num_tokens`: The total number of tokens processed so far.
* `loss`: The average cross-entropy loss computed over non-masked tokens in the current logging interval.
* `entropy`: The average entropy of the model's predicted token distribution over non-masked tokens.
* `mean_token_accuracy`: The proportion of non-masked tokens for which the model’s top-1 prediction matches the ground truth token.
* `learning_rate`: The current learning rate, which may change dynamically if a scheduler is used.
* `grad_norm`: The L2 norm of the gradients, computed before gradient clipping.

### Expected dataset type and format

SFT supports both [language modeling](https://huggingface.co/docs/trl/dataset_formats#language-modeling) and [prompt-completion]([dataset_formats](https://huggingface.co/docs/trl/dataset_formats)#prompt-completion) datasets. The [`SFTTrainer`] is compatible with both [standard]([dataset_formats](https://huggingface.co/docs/trl/dataset_formats)#standard) and [conversational]([dataset_formats](https://huggingface.co/docs/trl/dataset_formats)#conversational) dataset formats. When provided with a conversational dataset, the trainer will automatically apply the chat template to the dataset.

```python
# Standard language modeling
{"text": "The sky is blue."}

# Conversational language modeling
{"messages": [{"role": "user", "content": "What color is the sky?"},
              {"role": "assistant", "content": "It is blue."}]}

# Standard prompt-completion
{"prompt": "The sky is",
 "completion": " blue."}

# Conversational prompt-completion
{"prompt": [{"role": "user", "content": "What color is the sky?"}],
 "completion": [{"role": "assistant", "content": "It is blue."}]}
```

If your dataset is not in one of these formats, you can preprocess it to convert it into the expected format. Here is an example with the [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset:

```python
from datasets import load_dataset

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

def preprocess_function(example):
    return {
        "prompt": [{"role": "user", "content": example["Question"]}],
        "completion": [
            {"role": "assistant", "content": f"<think>{example['Complex_CoT']}</think>{example['Response']}"}
        ],
    }

dataset = dataset.map(preprocess_function, remove_columns=["Question", "Response", "Complex_CoT"])
print(next(iter(dataset["train"])))
```

```json
{
    "prompt": [
        {
            "content": "Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?",
            "role": "user",
        }
    ],
    "completion": [
        {
            "content": "<think>Okay, let's see what's going on here. We've got sudden weakness [...] clicks into place!</think>The specific cardiac abnormality most likely to be found in [...] the presence of a PFO facilitating a paradoxical embolism.",
            "role": "assistant",
        }
    ],
}
```

### Chat Templates in Training

We'll return briefly to chat templates in the context of training. Using chat templates correctly during training is crucial for model performance. Here are the key considerations and best practices:

#### Preprocessing and tokenization

During training, each example is expected to contain a **text field** or a **(prompt, completion)** pair, depending on the dataset format. For more details on the expected formats, see [Dataset formats](https://huggingface.co/docs/trl/dataset_formats).
The [`SFTTrainer`](https://huggingface.co/docs/trl/sft_trainer) tokenizes each input using the model's tokenizer. If both prompt and completion are provided separately, they are concatenated before tokenization.

#### Computing the loss

![sft_figure](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/sft_figure.png)

The loss used in SFT is the **token-level cross-entropy loss**, defined as:

$$
\mathcal{L}_{\text{SFT}}(\theta) = - \sum_{t=1}^{T} \log p_\theta(y_t \mid y_{<t}),
$$
  
where  \\( y_t \\) is the target token at timestep  \\( t \\), and the model is trained to predict the next token given the previous ones. In practice, padding tokens are masked out during loss computation.

### Supervised Fine-Tuning with TRL (Transformer Reinforcement Learning)

**TRL** is the go-to toolkit for training language models, built specifically for instruction tuning and alignment. It's what we'll use throughout this course.

#### Why TRL?

- Production ready: Used by major organizations and research labs.
- Comprehensive: Supports SFT, DPO, ORPO, PPO, and more advanced techniques.
- Efficient: Optimized for memory usage and training speed.
- Flexible: Works with any Hugging Face model.
- CLI support: Command-line tools for scalable training workflows.

#### Key Components

- **SFTTrainer**: The core class for supervised fine-tuning
- **SFTConfig**: Configuration management for training parameters
- **CLI Tools**: Command-line interface for production workflows
- **Integration**: Seamless integration with Hugging Face Hub, Trackio, Weights & Biases, and more

#### TRL's Architecture

TRL is built on top of the Hugging Face ecosystem:
- Transformers: Model loading and inference.
- Datasets: Data processing and management.
- Accelerate: Distributed training and optimization.
- PEFT: Parameter-efficient fine-tuning (LoRA, QLoRA).

This integrated approach means you get all the benefits of the Hugging Face ecosystem while using state-of-the-art training techniques.

> [!TIP]
> TRL versus other training libraries:
> - TRL: Specialized for LLM training, built for instruction tuning.
> - Transformers Trainer: General purpose, suitable for basic fine-tuning.
> - DeepSpeed: Focuses on large-scale distributed training.
> - Accelerate: Provides low-level distributed training primitives.
>
> TRL provides the best balance of ease-of-use and advanced features for SFT. For more details on training approaches, see the [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter11/3).


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import trackio as wandb

# Initialize experiment tracking
wandb.init(project="smolm3-sft", name="first-sft-run")

# Load SmolLM3 base model
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM3-3B-Base")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B-Base")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


* Trackio project initialized: smolm3-sft
* Trackio metrics logged to: /Users/abhisheksingh/.cache/huggingface/trackio


* Created new run: first-sft-run


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
dataset = load_dataset("HuggingFaceTB/smoltalk2_everyday_convs_think")

In [ ]:
config = SFTConfig(
    output_dir="./smolm3-finetuned",
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    max_steps=10000,
    report_to="trackio"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    args=config
)

trainer.train()